# Flux reconstruction, error indicators and adaptive decisions

This lesson starts with user-written UFL local/global equations. It then defines a physical coefficient record, reconstructs an H(div) flux, evaluates an estimator and chooses cells for refinement. Numerical algorithms remain in the package; no physical-model constructor selects the equations.

The primary case is $p=\sin(2\pi x)\sin(2\pi y)$, $K=I$, $f=8\pi^2p$, with homogeneous Dirichlet pressure on the unit square. The formulation follows [Harder, Paredes and Valentin (2013)](https://doi.org/10.1016/j.jcp.2013.03.019). Flux recovery and the four-term estimator follow [Barrenechea, Martins, Pereira and Valentin (2026)](https://doi.org/10.1137/24M1673073).

Install the native UFL backend as explained in the installation guide. The workflow is **meshes → local forms → global forms → assemble → solve → recovery → indicators → marking**.

In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Preserve this exact notebook as the numerical source checked by its records.
NOTEBOOK_RELATIVE = Path("notebooks/darcy/reconstruction_and_indicators.ipynb")
source_declaration = os.environ.get("PYMHM_NOTEBOOK_SOURCE")
source_candidates = (
    (Path(source_declaration),) if source_declaration else
    (Path.cwd() / NOTEBOOK_RELATIVE.name, Path.cwd() / NOTEBOOK_RELATIVE)
)
NOTEBOOK_SOURCE = next((path for path in source_candidates if path.is_file()), None)
if NOTEBOOK_SOURCE is None:
    raise FileNotFoundError(
        "Keep this notebook in the launch directory, or set PYMHM_NOTEBOOK_SOURCE "
        "to its actual file before starting the kernel."
    )
notebook_source_bytes = NOTEBOOK_SOURCE.read_bytes()

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/5072e36e43f04fcb78aa65bcadd12a801743be6c5fd416c939ca25f1962d5136/reconstruction_and_indicators-companion.zip"
COMPANION_SHA256 = "5072e36e43f04fcb78aa65bcadd12a801743be6c5fd416c939ca25f1962d5136"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/reconstruction_and_indicators.ipynb", directory=ROOT)

# Stage actual source bytes, including their independent provenance fingerprint.
workspace_notebook = ROOT / NOTEBOOK_RELATIVE
workspace_notebook.parent.mkdir(parents=True, exist_ok=True)
workspace_notebook.write_bytes(notebook_source_bytes)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ufl
from pymhm import (
    Equation, FaceSpace, LocalContext, MeshHierarchy, SkeletonSpace, TriangleMesh,
    assemble, bind_interface, bind_problem, columns, solve,
)
from pymhm.postprocessing.solutions import DarcySolution
from pymhm.recovery.moments import reconstruct_darcy_moments
from pymhm.estimators.darcy import estimate_darcy_error
from pymhm.adaptivity.darcy import mark_dorfler
from threadpoolctl import threadpool_limits


def exact_pressure(points: np.ndarray) -> np.ndarray:
    """Analytical pressure; it vanishes on every exterior edge."""
    return np.sin(2 * np.pi * points[:, 0]) * np.sin(2 * np.pi * points[:, 1])


def exact_gradient(points: np.ndarray) -> np.ndarray:
    """Independently differentiated physical pressure gradient."""
    x, y = 2 * np.pi * points.T
    return 2 * np.pi * np.column_stack((np.cos(x) * np.sin(y), np.sin(x) * np.cos(y)))


def source(points: np.ndarray) -> np.ndarray:
    """Negative Laplacian computed independently of the discrete operator."""
    return 8 * np.pi**2 * exact_pressure(points)
RUN_STUDY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## 1. Choose meshes and approximation spaces

For reconstruction degree $m=2$ and trace degree $\ell=1$, choose local degree $k=3$. This satisfies the two-dimensional restriction $k\ge\ell+2$ and $\ell\le m\le k$. The estimator additionally needs convex macrotriangles, a globally conforming fine partition, identity diffusion and homogeneous Dirichlet data. These are mathematical restrictions, not conventions inferred by the software.

In [ ]:
macro = TriangleMesh.unit_square(8)
local_meshes = tuple(macro.submesh(cell, 2) for cell in range(len(macro.cells)))
hierarchy = MeshHierarchy(macro, local_meshes)
skeleton = SkeletonSpace(macro, tuple(FaceSpace.uniform(1) for _ in macro.faces))
interface = bind_interface(skeleton, convention="normal")

## 2. Declare local equations

$$
\begin{aligned}
(\nabla p_T,\nabla v)_T+\langle\lambda_T,v\rangle_{\partial T}&=(f,v)_T,\\
-\langle p_T,\mu_T\rangle_{\partial T}&=g_T(\mu_T).
\end{aligned}
$$

The local volume form has a constant kernel. Its physical volume moment fixes the local complement and retains one mean per macrocell. The interface binding supplies normal incidence and numbering; the minus sign in the global test pairing is part of the formulation.

In [ ]:
def local_equations(local: LocalContext, degree: int = 3, forcing=None):
    """Declare primal diffusion, independent trace tests and physical mean."""
    space = local.native_space(degree=degree)
    p, v = ufl.TrialFunction(space.space), ufl.TestFunction(space.space)
    x = ufl.SpatialCoordinate(space.mesh)
    dx = ufl.Measure("dx", domain=space.mesh, metadata={"quadrature_degree": 20})
    forcing = (8 * np.pi**2 * ufl.sin(2 * np.pi * x[0]) * ufl.sin(2 * np.pi * x[1])
               if forcing is None else forcing(space.mesh))
    local.field("pressure", space)
    return local.equations(
        a=ufl.inner(ufl.grad(p), ufl.grad(v)) * dx,
        L=forcing * v * dx,
        b=local.trace_pairings(lambda phi, ds: phi * v * ds),
        c=local.trace_pairings(lambda phi, ds: -phi * p * ds, axis="rows"),
        kernel=np.ones((space.size, 1)), moments=columns(v * dx),
    )

## 3. Declare global boundary moments

$$
-\sum_T\langle p_T,\mu_T\rangle_{\partial T}
=-\langle p_D,\mu\rangle_{\partial\Omega},\qquad p_D=0.
$$

The Dirichlet load is therefore zero. Its sign is kept explicit so that changing the boundary data does not change the formulation accidentally.

In [ ]:
def global_equation(global_context):
    """Supply the weak Dirichlet moments in the bound interface layout."""
    boundary, fixed = global_context.boundary_data(0.0, order=8)
    assert not fixed
    return Equation(0, global_context.trace_load(-boundary))


problem = bind_problem(hierarchy, interface, local_equations,
                       global_equation=global_equation, retained=1)

## 4. Assemble, solve and name the physical result

Named fields preserve the native executed coefficient basis. `portable_coefficients` converts through its stored mapping; reshaping native vectors would not be equivalent. `DarcySolution` below is a data record for established reconstruction and estimator operations. Constructing it performs no PDE solve.

The `raw_residual` printed below is the original **global compatibility** diagnostic for trace and retained equations. It is distinct from every original local volume row and from a physical L2 error. The companion native original-row observer separately checks the same eight-subdivision P2/P0 and P3/P1 equations; their maximum relative local defects are 3.32e-15 and 1.04e-14, with full uncondensed defects relative to the physical load 5.67e-15 and 1.76e-14. The localized control also passes: 4.20e-15 and 6.12e-15. All use the unchanged 1e-10 equation gate.


In [ ]:
with threadpool_limits(1):
    system = assemble(problem)
    coefficients = solve(system)
pressure_fields = coefficients.field("pressure")
solution = DarcySolution(
    skeleton=skeleton, local_meshes=local_meshes,
    pressure=tuple(field.portable_coefficients for field in pressure_fields),
    flux=(), hybrid=coefficients, formulation="primal", permeability=1.0,
    source=source, degree=3, quadrature_order=10,
)
print({"original_equation_residual": coefficients.raw_residual,
       "pressure_L2": solution.l2_error(exact_pressure, order=12)})

## 5. Reconstruct a flux and distinguish its conservation tests

The canonical RT moment reconstruction preserves boundary normal moments from the skeleton, averages interior normal moments and preserves raw interior vector moments. It is not the energy-minimizing RT0 equilibration. Its divergence balance is against continuous macro-local tests, not individual discontinuous fine-cell constants.

$$
\begin{aligned}
\langle q_h\cdot n,\phi\rangle_E&=\text{declared normal moments},\\
(q_h,\psi)_t&=(-\nabla p_h,\psi)_t.
\end{aligned}
$$

In [ ]:
recovered = reconstruct_darcy_moments(solution, degree=2, quadrature_order=10)
normal_defect = max(np.max(abs(row)) for row in recovered.normal_flux_residuals())
continuous_defect = max(np.max(abs(row)) for row in recovered.continuous_moment_residuals())
fine_defect = max(np.max(abs(row)) for row in recovered.fine_conservation_residuals())
assert normal_defect < 1e-10
assert continuous_defect < 1e-10
print({"normal_moment_defect": normal_defect,
       "continuous_test_balance_defect": continuous_defect,
       "fine_cell_balance_defect_not_imposed": fine_defect,
       "recovered_flux_L2": recovered.flux_l2_error(lambda x: -exact_gradient(x), order=12)})

## 6. Evaluate the four-term indicator

$$
\eta^2=\sum_T\big[(\eta_{1,T}+\eta_{3,T}+\eta_{\mathrm{osc},T})^2
+\eta_{2,T}^2\big].
$$

The contributions measure flux defect, potential nonconformity, divergence-projection defect and source oscillation. The recovered potential is a separate continuous field. For this unit-diffusion case, compare the estimator with the independently integrated broken energy error; numerical quadrature does not provide an interval-certified bound.

In [ ]:
estimate = estimate_darcy_error(solution, homogeneous_dirichlet=True,
                                degree=2, quadrature_order=10)
energy_error = estimate.energy_error(exact_gradient, order=12)
assert max(estimate.equilibrium_defect) < 1e-10
assert estimate.total >= energy_error
print({"energy_error": energy_error, "estimator": estimate.total,
       "effectivity": estimate.total / energy_error})

## 7. Plot physical fields with the actual macro mesh

Each field panel has its own color scale. Numerical values and errors are sampled independently in each local mesh, preserving interface sides.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), layout="constrained")
all_exact, all_values, panels = [], [], []
for fine, field in zip(local_meshes, pressure_fields, strict=True):
    points = fine.points[fine.cells].mean(axis=1)
    exact, values = exact_pressure(points), field.evaluate(points)
    panels.append((fine, exact, values))
    all_exact.extend(exact)
    all_values.extend(values)
common = max(np.max(np.abs(all_exact)), np.max(np.abs(all_values)))
error_max = max(np.max(abs(values - exact)) for _, exact, values in panels)
for index, (ax, title) in enumerate(zip(axes, ("Analytical pressure", "MHM pressure", "Pressure error"), strict=True)):
    for fine, exact, values in panels:
        value = (exact, values, values - exact)[index]
        limit = error_max if index == 2 else common
        artist = ax.tripcolor(*fine.points.T, fine.cells, facecolors=value,
                             vmin=-limit, vmax=limit, cmap="coolwarm")
    for edge in macro.faces:
        ax.plot(*macro.points[edge].T, color="black", lw=0.5, alpha=0.7)
    ax.set(title=title, xlabel="x", ylabel="y", aspect="equal")
    fig.colorbar(artist, ax=ax)
plt.show()

## 8. Mark cells; choose the scale you want to improve

Dörfler marking selects a minimal sorted collection carrying the prescribed fraction of the squared indicator:

$$
\sum_{T\in\mathcal M}\eta_T^2\ge\theta\sum_T\eta_T^2.
$$

Marking is distinct from refining. Before changing a mesh, choose whether the dominant error comes from the macro partition, local approximation, or trace resolution. Refinement ancestry must preserve material and boundary markers. Rebuild the equations on the new spaces and compare error against work; a general adaptive sequence has no predetermined uniform-grid rate.

In [ ]:
marked = mark_dorfler(estimate.local_squared, theta=0.5)
assert estimate.local_squared[marked].sum() >= 0.5 * estimate.local_squared.sum()
fig, ax = plt.subplots(figsize=(5, 4), layout="constrained")
artist = ax.tripcolor(*macro.points.T, macro.cells, facecolors=estimate.local_squared,
                     edgecolors="black", cmap="viridis")
centers = macro.points[macro.cells[marked]].mean(axis=1)
ax.scatter(*centers.T, marker="x", color="white", s=60, label="Dörfler marked")
ax.set(xlabel="x", ylabel="y", title="Squared macro indicators", aspect="equal")
ax.legend()
fig.colorbar(artist, ax=ax)
plt.show()
print({"marked_macro_cells": np.flatnonzero(marked).tolist(), "marked_fraction":
       float(estimate.local_squared[marked].sum() / estimate.local_squared.sum())})

## 9. Refine a localized physical solution, rebuild and compare work

Uniform refinement first establishes the expected smooth-case order. Adaptivity
then uses a different smooth pressure with a localized peak,

$$
p(x,y)=x(1-x)y(1-y)\exp\!\left[-80\big((x-0.3)^2+(y-0.4)^2\big)\right].
$$

It vanishes on the entire boundary. The following NumPy source is differentiated
independently from the UFL source. Both define the same unit-diffusion operator.
This manufactured adaptive test uses the estimator of Barrenechea et al. (2026);
it is an analytical extension, rather than their heterogeneous-material example.
The error is compared against the exact gradient and against uniform meshes at
similar work. Adaptive mesh diameters are nonuniform: the horizontal coordinate
is the total number of retained and trace unknowns, not a fictitious uniform $H$.

In [ ]:
from functools import partial
from time import perf_counter
from pymhm.meshes.refinement import refine_triangles
from examples.tutorial_recovery_acquisition import recovery_diagnostics, recovery_display_samples, read_recovery_record


def solve_declared(macro, ell, source_values, forcing=None):
    """Assemble the already declared forms on caller-chosen meshes and spaces."""
    meshes = tuple(macro.submesh(i, 2) for i in range(len(macro.cells)))
    trace = SkeletonSpace(macro, tuple(FaceSpace.uniform(ell) for _ in macro.faces))
    problem = bind_problem(
        MeshHierarchy(macro, meshes), bind_interface(trace, convention="normal"),
        partial(local_equations, degree=ell + 2, forcing=forcing),
        global_equation=global_equation, retained=1,
    )
    with threadpool_limits(1):
        coefficients = solve(assemble(problem))
    result = DarcySolution(
        skeleton=trace, local_meshes=meshes,
        pressure=tuple(field.portable_coefficients for field in coefficients.field("pressure")),
        flux=(), hybrid=coefficients, formulation="primal", permeability=1.0,
        source=source_values, degree=ell + 2, quadrature_order=16,
    )
    return result


def localized_pressure(points):
    """A boundary-compatible smooth peak at (0.3, 0.4)."""
    x, y = points.T
    return x * (1-x) * y * (1-y) * np.exp(-80*((x-.3)**2 + (y-.4)**2))


def localized_gradient(points):
    """Independent first derivatives of the exact localized pressure."""
    x, y = points.T
    g = np.exp(-80*((x-.3)**2 + (y-.4)**2))
    X, Y = x*(1-x), y*(1-y)
    return g[:, None] * np.column_stack((Y*(1-2*x-160*(x-.3)*X), X*(1-2*y-160*(y-.4)*Y)))


def localized_source(points):
    """Independent negative Laplacian, rather than a discrete manufactured load."""
    x, y = points.T
    g = np.exp(-80*((x-.3)**2 + (y-.4)**2))
    X, Y = x*(1-x), y*(1-y)
    dxx = Y * (-2-320*(x-.3)*(1-2*x) + X*(25600*(x-.3)**2-160))
    dyy = X * (-2-320*(y-.4)*(1-2*y) + Y*(25600*(y-.4)**2-160))
    return -(dxx+dyy)*g


def localized_forcing(domain):
    """Differentiate the analytical UFL expression independently of NumPy."""
    x = ufl.SpatialCoordinate(domain)
    p = x[0]*(1-x[0])*x[1]*(1-x[1])*ufl.exp(-80*((x[0]-.3)**2+(x[1]-.4)**2))
    return -ufl.div(ufl.grad(p))


if RUN_STUDY:
    adaptive_rows = []
    adaptive_mesh = TriangleMesh.unit_square(4)
    for step in range(8):
        with threadpool_limits(1):
            adaptive_solution = solve_declared(adaptive_mesh, 1, localized_source, localized_forcing)
            row, adaptive_estimate = recovery_diagnostics(
                adaptive_solution, localized_pressure, localized_gradient,
            )
        row["step"] = step
        row["global_unknowns"] = row["trace_dofs"] + row["macro_cells"]
        if adaptive_rows:
            assert row["energy_error"] < adaptive_rows[-1]["energy_error"]
        adaptive_rows.append(row)
        print({key: row[key] for key in ("step", "macro_cells", "global_unknowns", "energy_error", "effectivity")})
        if step < 7:
            marked = mark_dorfler(adaptive_estimate.local_squared, theta=0.5)
            assert adaptive_estimate.local_squared[marked].sum() >= 0.5*adaptive_estimate.local_squared.sum()
            adaptive_mesh = refine_triangles(adaptive_mesh, marked).mesh
else:
    adaptive_data = read_recovery_record(
        ROOT/"examples/results/tutorial-methods/adaptivity-current.json",
        notebook=ROOT/"notebooks/darcy/reconstruction_and_indicators.ipynb",
    )
    adaptive_rows = adaptive_data["adaptive"]
    adaptive_mesh = TriangleMesh(adaptive_data["final_macro_points"], adaptive_data["final_macro_cells"])
    for row in adaptive_rows:
        print({key: row[key] for key in ("step", "macro_cells", "global_unknowns", "energy_error", "effectivity")})


## 10. Recover a conforming potential and a separately equilibrated RT0 flux

The Oswald potential averages incident finite-element nodal traces in the global continuous P3 space, then imposes the declared homogeneous boundary data. Its global coefficient vector supplies a single value at every shared node. This is a different field from the original broken pressure.

For strict fine-cell balance, instead solve the weighted RT0 minimization with the same physical source and a **degree-zero aligned** skeletal trace. Reassemble those newly declared spaces first; do not pass incompatible P1 trace data to the RT0 operator. The minimization is an available reconstruction, not an assertion of equivalence to the published RT2 moment operator.

In [ ]:
from pymhm.estimators.darcy import recover_potential
from pymhm.recovery.equilibrated import equilibrate_flux
from pymhm.fem.scalar.triangle import nodal_space

potential = recover_potential(solution, homogeneous_dirichlet=True)
_, potential_points = nodal_space(potential.mesh, potential.degree)
boundary_nodes = np.any(np.isclose(potential_points, 0.0) | np.isclose(potential_points, 1.0), axis=1)
assert np.max(abs(potential.values[boundary_nodes])) == 0.0
print({"conforming_potential_L2": potential.l2_error(exact_pressure, order=12),
       "potential_boundary_defect": float(np.max(abs(potential.values[boundary_nodes])))})

rt0_skeleton = SkeletonSpace(macro)
rt0_interface = bind_interface(rt0_skeleton, convention="normal")
rt0_problem = bind_problem(hierarchy, rt0_interface, local_equations,
                          global_equation=global_equation, retained=1)
with threadpool_limits(1):
    rt0_system = assemble(rt0_problem)
    rt0_coefficients = solve(rt0_system)
rt0_solution = DarcySolution(
    skeleton=rt0_skeleton, local_meshes=local_meshes,
    pressure=tuple(field.portable_coefficients for field in rt0_coefficients.field("pressure")),
    flux=(), hybrid=rt0_coefficients, formulation="primal", permeability=1.0,
    source=source, degree=3, quadrature_order=10,
)
equilibrated = equilibrate_flux(rt0_solution)
fine_balance = max(np.max(abs(row)) for row in equilibrated.conservation_residuals())
assert fine_balance < 1e-10
print({"equilibrated_fine_cell_balance": fine_balance,
       "equilibrated_Darcy_flux_L2": equilibrated.l2_error(lambda x: -exact_gradient(x), order=12)})

## 11. Establish the asymptotic regime before interpreting recovery

Barrenechea et al. (2026), Theorem 4.7 and Remark 5.1, give

$$
\begin{aligned}
\lVert q-q_h^{\mathrm{RT}}\rVert_{0,\Omega}&\le C(h^k+H^{\ell+1}+h^{\ell+1}),\\
\lVert f-\Pi_{\Omega,m}\operatorname{div}q_h^{\mathrm{RT}}\rVert_{0,\Omega}&\le Ch^{m+1}.
\end{aligned}
$$

Here $h=H/2$, $k=\ell+2$, and $m=2$. Thus the recovered physical flux has
order one for $\ell=0$ and two for $\ell=1$, while its **projected** divergence
has order three. The raw divergence is a distinct quantity and is reported
separately. Local degree, RT degree and mesh ratios remain fixed throughout.
The estimator's reliability bound and stable effectivity are checked at every
level; no theorem requires a universal gain in recovered pressure error.

The acquisition helper only measures already defined fields. It does not choose
the operator or spaces. Set `PYMHM_NOTEBOOK_STUDY=1` to execute every study solve;
otherwise the same current numerical record supplies the convergence figures.

In [ ]:
from examples.tutorial_recovery_acquisition import write_recovery_record, recovery_provenance
from examples.tutorial_convergence import plot_method_series, asymptotic_summary

study_path = ROOT / "examples/results/tutorial-methods/recovery-current.json"
RUN_STUDY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"
source_paths = [ROOT/"notebooks/darcy/reconstruction_and_indicators.ipynb", ROOT/"examples/tutorial_recovery_acquisition.py", ROOT/"pixi.lock"]
if RUN_STUDY:
    executed_sources = recovery_provenance(ROOT, source_paths)
    uniform_rows = []
    for ell in (0, 1):
        for n in (4, 8, 16, 32):
            started = perf_counter()
            with threadpool_limits(1):
                state = solve_declared(TriangleMesh.unit_square(n), ell, source)
                row, state_estimate = recovery_diagnostics(state, exact_pressure, exact_gradient)
            row.update(macro_resolution=n, wall_seconds=perf_counter()-started)
            uniform_rows.append(row)
            print({key: row[key] for key in ("trace_degree", "macro_resolution", "energy_error", "recovered_flux_l2", "projected_divergence_l2", "effectivity", "wall_seconds")})
    write_recovery_record(
        study_path, root=ROOT,
        source_paths=source_paths, source_manifest=executed_sources,
        data={"case": "sin(2*pi*x)sin(2*pi*y), identity diffusion, zero Dirichlet",
              "scope": "Current native UFL analytical study of the published section 6.1 data",
              "reference": "10.1137/24M1673073", "local_subdivisions": 2,
              "assembly_quadrature_degree": 20, "uniform": uniform_rows},
    )
else:
    uniform_rows = read_recovery_record(study_path, notebook=ROOT/"notebooks/darcy/reconstruction_and_indicators.ipynb")["uniform"]

for ell in (0, 1):
    rows = [row for row in uniform_rows if row["trace_degree"] == ell]
    h = np.array([row["macro_diameter"] for row in rows])
    errors = {"recovered Darcy flux L2": np.array([row["recovered_flux_l2"] for row in rows]),
              "broken energy error": np.array([row["energy_error"] for row in rows]),
              "projected divergence L2": np.array([row["projected_divergence_l2"] for row in rows])}
    series = {"method": f"RT2 recovery, trace degree {ell}", "sizes": h.tolist(),
              "errors": {key: values.tolist() for key, values in errors.items()},
              "rates": {key: (np.log(values[:-1]/values[1:])/np.log(h[:-1]/h[1:])).tolist() for key, values in errors.items()},
              "expected": {"recovered Darcy flux L2": ell+1, "broken energy error": ell+1, "projected divergence L2": 3},
              "refinement": "Macro diameter H, with h=H/2",
              "spaces": f"P{ell+2} local pressure; P{ell} normal trace; RT2 reconstruction",
              "rate_provenance": "Barrenechea et al. (2026), Theorem 4.7 and Remark 5.1"}
    print(asymptotic_summary(series))
    plot_method_series(series)
    plt.show()

## 12. Compare a classical refined conforming Galerkin reference

The baseline is assembled independently on a global continuous P3 space.
The local MHM equations are not reused as its assembled matrix. Three global
meshes verify its own refinement error against the analytical pressure and
flux; the analytical solution remains the exact comparator. Each physical norm
is integrated with two quadrature orders. This separates reference accuracy
from the approximation quality of the coarse-interface method.

In [ ]:
from examples.introduction.scalar import native_scalar_space, dirichlet_solve
from examples.introduction.transport import scalar_error_norms
from pymhm.core.equations import compile_form
from pymhm.fem.scalar.triangle import nodal_space

classical_rows = []
for n in (16, 32, 64):
    fine = TriangleMesh.unit_square(n)
    domain, V, mapping = native_scalar_space(fine, 3)
    p, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    x = ufl.SpatialCoordinate(domain)
    dx = ufl.Measure("dx", domain=domain, metadata={"quadrature_degree": 20})
    f = 8*np.pi**2*ufl.sin(2*np.pi*x[0])*ufl.sin(2*np.pi*x[1])
    matrix = compile_form(ufl.inner(ufl.grad(p), ufl.grad(v))*dx)
    load = compile_form(f*v*dx)
    _, nodes = nodal_space(fine, 3)
    exterior = np.flatnonzero(np.any(np.isclose(nodes, 0)|np.isclose(nodes, 1), axis=1))
    with threadpool_limits(1):
        values = dirichlet_solve(matrix, load, mapping[exterior], np.zeros(len(exterior)))[mapping]
        low = scalar_error_norms((fine,), (values,), 3, exact_pressure, exact_gradient, 1.0, order=12)
        high = scalar_error_norms((fine,), (values,), 3, exact_pressure, exact_gradient, 1.0, order=16)
    assert abs(low["flux_l2"]-high["flux_l2"]) < 1e-7*high["flux_l2"]
    classical_rows.append({"n": n, "unknowns": len(nodes), **high})
    print(classical_rows[-1])

## 13. Inspect the finest recovered fields and the adaptive error/work curve

The analytical pressure, original MHM pressure, signed pressure error, analytical
Darcy-flux magnitude, recovered RT2 Darcy-flux magnitude and vector-flux error
are sampled separately on each fine triangle. The actual macro partition is
shown in every panel. Convergence uses integrated errors, not these samples.
The final adaptive mesh and local indicators show where the localized peak
requires extra resolution. The uniform comparator solves the same localized
operator and data at a fixed polynomial degree.

In [ ]:
from examples.tutorial_recovery_plots import plot_recovery_fields, plot_adaptive_study, plot_estimator_study

with threadpool_limits(1):
    display_solution = solve_declared(TriangleMesh.unit_square(16), 1, source)
    display_row, display_estimate = recovery_diagnostics(display_solution, exact_pressure, exact_gradient)
    display_samples = recovery_display_samples(display_solution, display_estimate, exact_pressure, exact_gradient)
plot_recovery_fields(display_samples)
plt.show()
plot_estimator_study(uniform_rows)
plt.show()

if RUN_STUDY:
    localized_uniform = []
    for n in (4, 8, 16, 32):
        with threadpool_limits(1):
            state = solve_declared(TriangleMesh.unit_square(n), 1, localized_source, localized_forcing)
            row, _ = recovery_diagnostics(state, localized_pressure, localized_gradient)
        row["global_unknowns"] = row["trace_dofs"] + row["macro_cells"]
        localized_uniform.append(row)
    final_indicators = adaptive_estimate.local_squared
else:
    localized_uniform = adaptive_data["uniform"]
    final_indicators = np.asarray(adaptive_data["final_indicators"])
plot_adaptive_study(adaptive_rows, localized_uniform, adaptive_mesh, final_indicators)
plt.show()

## 14. Use another multiscale problem as a local operator

A physical two-level hierarchy replaces each outer local solve by four child
macrotriangles. Leaf pressure remains P3 on two local edge subdivisions and
child flux traces remain P1. Two independent P1 segments on each parent face
span exactly the two child boundary traces. The automatic restriction preserves
normal signs and segment coordinates; it is not an approximate projection.
The child problem leaves exterior boundary values to its parent.

The constant child pressure mode is zero on trace coordinates and one on each
child retained mean. Its physical moment uses child areas. Therefore each level
retains the same physical volume average. The finest named pressure fields are
obtained with `recursive=True`; no leaf is solved a second time during recovery.
Nested condensation does not introduce a new convergence theorem. When its
trace spaces span the same leaf discretization, it must reproduce the flattened
MHM solution and inherit that discretization's physical error rate.

In [ ]:
def recursive_provider(local: LocalContext):
    """Declare a child global problem inside this parent local macrotriangle."""
    child_macro = local.mesh
    leaf_meshes = tuple(child_macro.submesh(i, 2) for i in range(len(child_macro.cells)))
    child_trace = SkeletonSpace(child_macro, tuple(FaceSpace.uniform(1) for _ in child_macro.faces))
    child = bind_problem(
        MeshHierarchy(child_macro, leaf_meshes), bind_interface(child_trace, convention="normal"),
        local_equations, retained=1,
    )
    kernel = np.r_[np.zeros(child_trace.size), np.ones(len(child_macro.cells))][:, None]
    moments = np.r_[np.zeros(child_trace.size), child_macro.areas][:, None]
    return local.nested(child, kernel=kernel, moments=moments)


if RUN_STUDY:
    recursive_rows = []
    for n in (2, 4, 8, 16):
        outer = TriangleMesh.unit_square(n)
        children = tuple(outer.submesh(i, 2) for i in range(len(outer.cells)))
        outer_trace = SkeletonSpace(outer, tuple(FaceSpace.uniform(1, 2) for _ in outer.faces))
        recursive_problem = bind_problem(
            MeshHierarchy(outer, children), bind_interface(outer_trace, convention="normal"),
            recursive_provider, global_equation=global_equation, retained=1,
        )
        with threadpool_limits(1):
            recursive_coefficients = solve(assemble(recursive_problem))
            leaves = recursive_coefficients.field("pressure", recursive=True)
            leaf_meshes = tuple(field.mesh for field in leaves)
            low = scalar_error_norms(leaf_meshes, leaves, 3, exact_pressure, exact_gradient, 1.0, order=12)
            high = scalar_error_norms(leaf_meshes, leaves, 3, exact_pressure, exact_gradient, 1.0, order=16)
        assert abs(low["flux_l2"]-high["flux_l2"]) < 1e-7*high["flux_l2"]
        assert recursive_coefficients.raw_residual < 1e-10
        assert max(child.raw_residual for child in recursive_coefficients.children) < 1e-10
        # Flatten the identical child macro mesh, including its actual diagonals.
        points = np.concatenate([child.points for child in children])
        _, first, inverse = np.unique(points, axis=0, return_index=True, return_inverse=True)
        cells, offset = [], 0
        for child in children:
            cells.extend(inverse[offset+child.cells])
            offset += len(child.points)
        flat_mesh = TriangleMesh(points[first], np.array(cells))
        with threadpool_limits(1):
            flat = solve_declared(flat_mesh, 1, source)
            flat_fields = flat.hybrid.field("pressure")
            discrepancy = max(float(np.max(abs(a.portable_coefficients-b.portable_coefficients)))
                              for a, b in zip(leaves, flat_fields, strict=True))
        assert discrepancy < 1e-9
        row = {"outer_resolution": n, "child_diameter": np.sqrt(2)/(2*n),
               "outer_trace_dofs": outer_trace.size, "leaf_macro_cells": len(flat_mesh.cells),
               "physical_errors": high, "flattened_coefficient_difference": discrepancy,
               "original_equation_residual": recursive_coefficients.raw_residual}
        recursive_rows.append(row)
        print(row)

else:
    recursive_rows = read_recovery_record(
        ROOT/"examples/results/tutorial-methods/recursive-current.json",
        notebook=ROOT/"notebooks/darcy/reconstruction_and_indicators.ipynb",
    )["recursive"]
    for row in recursive_rows:
        print(row)

h = np.array([row["child_diameter"] for row in recursive_rows])
errors = np.array([row["physical_errors"]["flux_l2"] for row in recursive_rows])
series = {"method": "Two-level recursive MHM", "sizes": h.tolist(),
          "errors": {"Darcy flux L2": errors.tolist()},
          "rates": {"Darcy flux L2": (np.log(errors[:-1]/errors[1:])/np.log(h[:-1]/h[1:])).tolist()},
          "expected": {"Darcy flux L2": 2}, "refinement": "Child macro diameter, fixed two-level ratio",
          "spaces": "P3 leaf pressure, P1 child traces, two P1 segments per parent face",
          "rate_provenance": "Exact equality to flattened primal MHM; Harder et al. (2013)"}
print(asymptotic_summary(series))
plot_method_series(series)
plt.show()

## References and extensions

- Harder, Paredes and Valentin (2013), *A family of Multiscale Hybrid-Mixed finite element methods for the Darcy equation with rough coefficients*, [DOI](https://doi.org/10.1016/j.jcp.2013.03.019).
- Barrenechea, Martins, Pereira and Valentin (2026), [DOI](https://doi.org/10.1137/24M1673073).

For a declared SPD coefficient, use the energy-normalized estimator and supply certified ellipticity bounds when required. RT0 energy-minimizing equilibration instead imposes fine-cell averages and needs degree-zero trace segments aligned with fine boundary faces. Flow and elasticity indicators have their own hypotheses. The corresponding tutorials explain these contracts rather than silently applying the scalar indicator to other equations.